In [ ]:
# Leakage Group Construction

# This notebook identifies potentially related vulnerability records before dataset partitioning.
# Normalised descriptions and shared CVE identifiers are used to construct groups that must remain
# together during train, validation, and test splitting.

In [1]:
import re
import unicodedata

import pandas as pd

from src.config import (
    BDU_COL,
    LABEL_COL,
    ID_COL,
    GROUP_COL,
    DOMAIN_COL,
    RU_LEAKAGE_FILE,
    NONRU_LEAKAGE_FILE,
    RESULTS_TABLES_DIR,
    ensure_project_directories,
)

from src.data import load_modelling_data

ensure_project_directories()

ru, nonru = load_modelling_data()

DESCRIPTION_COL = "Описание уязвимости"
CVE_COL = "Идентификаторы других систем описаний уязвимости"

print("Russian:", ru.shape)
print("Non-Russian:", nonru.shape)

Russian: (14451, 32)
Non-Russian: (41663, 32)


In [ ]:
## Prepare Record Identifiers and Normalised Descriptions

# Domain labels and unique modelling-row identifiers are added.
# Vulnerability descriptions are normalised so that equivalent descriptions
# with minor formatting differences can be detected.

In [2]:
ru = ru.copy()
nonru = nonru.copy()

ru[DOMAIN_COL] = "Russian"
nonru[DOMAIN_COL] = "Non-Russian"

ru[ID_COL] = [
    f"RU_{i:05d}"
    for i in range(len(ru))
]

nonru[ID_COL] = [
    f"NR_{i:05d}"
    for i in range(len(nonru))
]


def normalize_description(text):

    if pd.isna(text):
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.lower()
    text = text.replace("ё", "е")

    text = re.sub(
        r"[–—−]",
        "-",
        text
    )

    text = (
        text
        .replace("«", '"')
        .replace("»", '"')
        .replace("“", '"')
        .replace("”", '"')
        .replace("„", '"')
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


ru["description_normalized"] = (
    ru[DESCRIPTION_COL]
    .apply(normalize_description)
)

nonru["description_normalized"] = (
    nonru[DESCRIPTION_COL]
    .apply(normalize_description)
)

In [ ]:
## Quantify Description Duplicates

# Exact and normalised description duplicates are counted separately for the Russian
# and non-Russian datasets to characterise repeated observations.

In [3]:
duplicate_summary = pd.DataFrame({

    "domain": [
        "Russian",
        "Non-Russian"
    ],

    "exact_duplicate_records": [
        ru.duplicated(
            subset=[DESCRIPTION_COL],
            keep="first"
        ).sum(),

        nonru.duplicated(
            subset=[DESCRIPTION_COL],
            keep="first"
        ).sum()
    ],

    "normalised_duplicate_records": [
        ru.duplicated(
            subset=["description_normalized"],
            keep="first"
        ).sum(),

        nonru.duplicated(
            subset=["description_normalized"],
            keep="first"
        ).sum()
    ]
})

duplicate_summary

,domain,exact_duplicate_records,normalised_duplicate_records
0,Russian,920,970
1,Non-Russian,8373,8595


In [ ]:
## Extract CVE Identifiers

# CVE identifiers are extracted from the external vulnerability-reference field
# and standardised for use in leakage-group construction.

In [5]:
CVE_PATTERN = re.compile(
    r"\bCVE-\d{4}-\d{4,7}\b",
    flags=re.IGNORECASE
)


def extract_cves(value):

    if pd.isna(value):
        return []

    return sorted({
        cve.upper()
        for cve
        in CVE_PATTERN.findall(str(value))
    })


ru["cve_list"] = (
    ru[CVE_COL]
    .apply(extract_cves)
)

nonru["cve_list"] = (
    nonru[CVE_COL]
    .apply(extract_cves)
)

ru["cve_ids"] = (
    ru["cve_list"]
    .apply(lambda x: "; ".join(x))
)

nonru["cve_ids"] = (
    nonru["cve_list"]
    .apply(lambda x: "; ".join(x))
)

In [ ]:
## Measure Cross-Domain Overlap

# Normalised descriptions and CVE identifiers occurring in both domains are identified.
# These overlaps indicate records that could otherwise introduce information leakage into cross-domain experiments.

In [6]:
shared_descriptions = (
    set(ru["description_normalized"])
    &
    set(nonru["description_normalized"])
)

shared_descriptions.discard("")


ru_cves = {
    cve
    for values in ru["cve_list"]
    for cve in values
}

nonru_cves = {
    cve
    for values in nonru["cve_list"]
    for cve in values
}

shared_cves = (
    ru_cves
    &
    nonru_cves
)


print(
    "Shared normalised descriptions:",
    len(shared_descriptions)
)

print(
    "Shared CVE identifiers:",
    len(shared_cves)
)

Shared normalised descriptions: 78
Shared CVE identifiers: 60


In [ ]:
## Initialise Leakage-Group Construction

# The two domains are combined and a union-find structure is initialised.
# This allows records connected through either duplicate descriptions or shared CVE identifiers
# to be represented as a single connected group.

In [7]:
all_records = pd.concat(
    [ru, nonru],
    ignore_index=True
)

assert len(all_records) == 56114


class UnionFind:

    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):

        if self.parent[x] != x:
            self.parent[x] = self.find(
                self.parent[x]
            )

        return self.parent[x]

    def union(self, a, b):

        a = self.find(a)
        b = self.find(b)

        if a == b:
            return

        if self.rank[a] < self.rank[b]:
            a, b = b, a

        self.parent[b] = a

        if self.rank[a] == self.rank[b]:
            self.rank[a] += 1


uf = UnionFind(
    len(all_records)
)

In [ ]:
## Link Related Vulnerability Records

# Records sharing the same normalised description or CVE identifier are connected.
# Transitive relationships are preserved, so indirectly related records are assigned to the same leakage group.

In [8]:
description_first_seen = {}

for idx, description in enumerate(
    all_records["description_normalized"]
):

    if not description:
        continue

    if description in description_first_seen:
        uf.union(
            idx,
            description_first_seen[description]
        )
    else:
        description_first_seen[
            description
        ] = idx


cve_first_seen = {}

for idx, cves in enumerate(
    all_records["cve_list"]
):

    for cve in cves:

        if cve in cve_first_seen:
            uf.union(
                idx,
                cve_first_seen[cve]
            )
        else:
            cve_first_seen[cve] = idx

In [ ]:
## Assign Leakage Groups

# A unique leakage-group identifier is assigned to every connected component.
# Groups spanning both domains and groups containing more than one CWE label are additionally flagged for later analysis.

In [9]:
root_to_group = {}
group_numbers = []

for i in range(len(all_records)):

    root = uf.find(i)

    if root not in root_to_group:
        root_to_group[root] = (
            len(root_to_group) + 1
        )

    group_numbers.append(
        root_to_group[root]
    )


all_records[GROUP_COL] = [
    f"LG_{number:05d}"
    for number in group_numbers
]


group_domain_count = (
    all_records
    .groupby(GROUP_COL)[DOMAIN_COL]
    .nunique()
)

cross_domain_groups = set(
    group_domain_count[
        group_domain_count > 1
    ].index
)


group_cwe_count = (
    all_records
    .groupby(GROUP_COL)[LABEL_COL]
    .nunique()
)

ambiguous_groups = set(
    group_cwe_count[
        group_cwe_count > 1
    ].index
)


all_records["cross_domain_group"] = (
    all_records[GROUP_COL]
    .isin(cross_domain_groups)
)

all_records["ambiguous_cwe_group"] = (
    all_records[GROUP_COL]
    .isin(ambiguous_groups)
)


print(
    "Total leakage groups:",
    all_records[GROUP_COL].nunique()
)

print(
    "Cross-domain groups:",
    len(cross_domain_groups)
)

print(
    "Groups with >1 CWE:",
    len(ambiguous_groups)
)

Total leakage groups: 42316
Cross-domain groups: 106
Groups with >1 CWE: 299


In [ ]:
## Summarise and Map Leakage Groups

# Group-level statistics are calculated and the resulting leakage-group identifiers
# and flags are mapped back to the original Russian and non-Russian datasets.

In [10]:
group_summary = (
    all_records
    .groupby(GROUP_COL)
    .agg(
        n_records=(ID_COL, "size"),
        n_domains=(DOMAIN_COL, "nunique"),
        n_cwe=(LABEL_COL, "nunique"),
        russian_records=(
            DOMAIN_COL,
            lambda x: (x == "Russian").sum()
        ),
        nonrussian_records=(
            DOMAIN_COL,
            lambda x: (x == "Non-Russian").sum()
        )
    )
    .reset_index()
)

group_summary["cross_domain"] = (
    group_summary["n_domains"] > 1
)

group_summary["ambiguous_cwe"] = (
    group_summary["n_cwe"] > 1
)


mapping = (
    all_records[
        [
            ID_COL,
            GROUP_COL,
            "cross_domain_group",
            "ambiguous_cwe_group",
        ]
    ]
    .set_index(ID_COL)
)


for dataframe in [ru, nonru]:

    dataframe[GROUP_COL] = (
        dataframe[ID_COL]
        .map(mapping[GROUP_COL])
    )

    dataframe["cross_domain_group"] = (
        dataframe[ID_COL]
        .map(mapping["cross_domain_group"])
    )

    dataframe["ambiguous_cwe_group"] = (
        dataframe[ID_COL]
        .map(mapping["ambiguous_cwe_group"])
    )


assert ru[GROUP_COL].notna().all()
assert nonru[GROUP_COL].notna().all()

In [ ]:
## Export Leakage-Controlled Data

# The datasets containing leakage-group identifiers are saved together
# with summary tables describing duplicates and group structure.

In [12]:
ru.drop(
    columns=["cve_list"],
    errors="ignore"
).to_excel(
    RU_LEAKAGE_FILE,
    index=False
)

nonru.drop(
    columns=["cve_list"],
    errors="ignore"
).to_excel(
    NONRU_LEAKAGE_FILE,
    index=False
)


duplicate_summary.to_csv(
    RESULTS_TABLES_DIR
    / "duplicate_summary.csv",
    index=False
)

group_summary.to_csv(
    RESULTS_TABLES_DIR
    / "leakage_group_summary.csv",
    index=False
)


print("Leakage-group datasets saved.")

Leakage-group datasets saved.
